# Photos Library Duplicate Cleanup Notebook

Report-only v1. This notebook does **not** delete anything.

Run order:
1. Run configuration.
2. Load/build inventory.
3. Fill identity fields.
4. Group and analyze duplicate candidates.
5. Write permanent operation report.

Helper functions live in `photos_duplicate_cleanup_helpers.py` so the notebook stays readable.


In [1]:
# ============================================================
# Cell 1. Configuration
# ============================================================

from pathlib import Path
import os
import sys
import json
from datetime import datetime

from explorephotoslibrary import *

PROJECT_ROOT = Path("/Users/huohsien/workspace/python/explore_photos_library")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ------------------------------------------------------------
# Photos Library registry
# ------------------------------------------------------------

PHOTOS_LIBRARY_PATHS = {
    "current_default": (
        "/Users/huohsien/Pictures/"
        "Photos Library.photoslibrary"
    ),
    "backup_20250317": (
        "/Volumes/NEW-PRO-G40--20250315/"
        "Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary"
    ),
    "test": (
        "/Users/huohsien/Pictures/"
        "test.photoslibrary"
    ),
}

# ------------------------------------------------------------
# Choose target library here.
#
# Valid examples:
#   "current_default"
#   "backup_20250317"
#   "test"
# ------------------------------------------------------------

TARGET_LIBRARY_ID = "backup_20250317"

if TARGET_LIBRARY_ID not in PHOTOS_LIBRARY_PATHS:
    raise KeyError(
        f"Unknown TARGET_LIBRARY_ID: {TARGET_LIBRARY_ID!r}\n"
        f"Available library ids: {sorted(PHOTOS_LIBRARY_PATHS)}"
    )

LIBRARY_ID = TARGET_LIBRARY_ID
LIBRARY_PATH = Path(PHOTOS_LIBRARY_PATHS[TARGET_LIBRARY_ID])

CACHE_DIR = PROJECT_ROOT / "data" / "inventory_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_NAME = LIBRARY_ID
INVENTORY_CACHE_PATH = CACHE_DIR / f"{CACHE_NAME}.inventory.pkl.gz"

REPORTS_ROOT = PROJECT_ROOT / "IMPORTANT_Photos_Library_Critical_Operation_Reports"
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
REPORT_DIR = REPORTS_ROOT / f"{RUN_TIMESTAMP}__Photos_Library_Duplicate_Cleanup__{LIBRARY_ID}"

print("TARGET_LIBRARY_ID:", TARGET_LIBRARY_ID)
print("LIBRARY_ID:", LIBRARY_ID)
print("LIBRARY_PATH:", LIBRARY_PATH)
print("LIBRARY_PATH exists:", LIBRARY_PATH.exists())
print("INVENTORY_CACHE_PATH:", INVENTORY_CACHE_PATH)
print("REPORT_DIR will be created only when writing report:", REPORT_DIR)

if not LIBRARY_PATH.exists():
    raise FileNotFoundError(f"Photos Library path does not exist: {LIBRARY_PATH}")

TARGET_LIBRARY_ID: backup_20250317
LIBRARY_ID: backup_20250317
LIBRARY_PATH: /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary
LIBRARY_PATH exists: True
INVENTORY_CACHE_PATH: /Users/huohsien/workspace/python/explore_photos_library/data/inventory_cache/backup_20250317.inventory.pkl.gz
REPORT_DIR will be created only when writing report: /Users/huohsien/workspace/python/explore_photos_library/IMPORTANT_Photos_Library_Critical_Operation_Reports/20260608-202149__Photos_Library_Duplicate_Cleanup__backup_20250317


In [2]:
# ============================================================
# Cell 2. Load or build inventory
# ============================================================

import osxphotos

FORCE_REBUILD_INVENTORY = True

if INVENTORY_CACHE_PATH.exists() and not FORCE_REBUILD_INVENTORY:
    inventory = load_inventory_cache(
        cache_name=CACHE_NAME,
        cache_dir=CACHE_DIR,
    )
    print("Loaded inventory cache:", INVENTORY_CACHE_PATH)
else:
    print("Building inventory from Photos Library:")
    print(LIBRARY_PATH)

    photosdb = osxphotos.PhotosDB(dbfile=str(LIBRARY_PATH))
    osx_assets = photosdb.photos()

    inventory = build_inventory(osx_assets)

    save_inventory_cache(
        inventory=inventory,
        cache_name=CACHE_NAME,
        cache_dir=CACHE_DIR,
    )

    print("Saved inventory cache:", INVENTORY_CACHE_PATH)

print_inventory_summary(inventory)

Building inventory from Photos Library:
/Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary
processed assets: 10000
processed assets: 20000
processed assets: 30000
processed assets: 40000
processed assets: 50000
processed assets: 60000
processed assets: 70000
saved inventory cache: /Users/huohsien/workspace/python/explore_photos_library/data/inventory_cache/backup_20250317.inventory.pkl.gz
elapsed seconds: 0.89
Saved inventory cache: /Users/huohsien/workspace/python/explore_photos_library/data/inventory_cache/backup_20250317.inventory.pkl.gz
inventory assets: 71596
inventory albums: 5172
inventory folders: 35
movies: 6238
hidden: 0
favorites: 699
descriptions: 727
keywords: 23756


In [3]:
# ============================================================
# Cell 3. Fill identity fields
# ============================================================

fill_duplicate_cleanup_identity_fields(inventory)


Filled identity fields
asset count: 71596
base_id filled: 71596
unique_id filled: 71596
adjustment_signature filled: 22222
edited_duration_seconds filled: 1054

First 3 assets after fill:
--------------------------------------------------------------------------------
original_filename: IMG_0461.HEIC
date: 2021-06-16T03:15:43.252628+08:00
path: /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/2/2D5205A6-1AC3-40E3-B44E-97E751D968A8.heic
path_edited: None
file_size_bytes: 1579915
edited_duration_seconds: None
adjustment_signature: (3024, 3024)
base_id: ('IMG_0461.HEIC', '06-16 03:15:43.252628', 1579915, (3024, 3024))
unique_id: (('IMG_0461.HEIC', '06-16 03:15:43.252628', 1579915, (3024, 3024)), (('description', None), ('keywords', ()), ('favorite', False), ('hidden', False), ('album_titles', ('嗨吃家 酸辣粉 - 一箱12盒。說什麼台灣的蕃薯做的，結果上面寫簡體字原來是統一企業在河南設廠，難道是把台灣的蕃薯拿去中國做粉？味道不知道怎麼樣... 很小碗是可以確定的！用過期九個月小肥羊麻辣鍋

In [4]:
# ============================================================
# Cell 4. Group duplicate candidates
# ============================================================

unique_id_groups, assets_without_unique_id = group_assets_by_field(
    inventory,
    "photo_library_asset_unique_id",
)

duplicate_candidate_groups = {
    unique_id: group
    for unique_id, group in unique_id_groups.items()
    if len(group) > 1
}

print("assets:", len(inventory["assets"]))
print("generated unique_id count:", len(unique_id_groups))
print("assets without unique_id:", len(assets_without_unique_id))
print("duplicate candidate group count:", len(duplicate_candidate_groups))
print("duplicate candidate asset count:", sum(len(group) for group in duplicate_candidate_groups.values()))

if assets_without_unique_id:
    print()
    print("First assets without unique_id:")
    for asset in assets_without_unique_id[:10]:
        print(
            asset.get("original_filename"),
            asset.get("uuid"),
            asset.get("asset_scope"),
            asset.get("path"),
        )


assets: 71596
generated unique_id count: 71583
assets without unique_id: 0
duplicate candidate group count: 13
duplicate candidate asset count: 26


In [5]:
# ============================================================
# Cell 5. Analyze duplicate candidates
# ============================================================

duplicate_analysis = analyze_duplicate_candidate_groups(duplicate_candidate_groups)


Analyzing group 1/13
Analyzing group 10/13
Analyzing group 13/13
analysis group count: 13
elapsed seconds: 40.804


In [6]:
# ============================================================
# Cell 6. Summary
# ============================================================

status_counts = count_records_by_status(duplicate_analysis)

delete_candidate_count = sum(
    len(record.get("delete_candidates") or [])
    for record in duplicate_analysis
)

print("status counts:")
for status, count in status_counts.items():
    print(f"  {status}: {count}")

print("delete candidate asset count:", delete_candidate_count)

print()
print("First deletable duplicate groups:")
printed = 0

for record in duplicate_analysis:
    if record.get("status") != "DELETABLE_DUPLICATE":
        continue

    print("-" * 80)
    print("reason:", record.get("reason"))
    print("asset_count:", record.get("asset_count"))
    print("keep_assets:", len(record.get("keep_assets") or []))
    print("delete_candidates:", len(record.get("delete_candidates") or []))

    for asset in (record.get("keep_assets") or []):
        print("  KEEP:", asset["original_filename"], asset["date_added"], asset["path"])

    for asset in (record.get("delete_candidates") or []):
        print("  DELETE:", asset["original_filename"], asset["date_added"], asset["path"])

    printed += 1

    if printed >= 10:
        print("... more groups not printed")
        break


status counts:
  DELETABLE_DUPLICATE: 13
delete candidate asset count: 13

First deletable duplicate groups:
--------------------------------------------------------------------------------
reason: SAME_UNIQUE_ID_AND_SAME_SHA256
asset_count: 2
keep_assets: 1
delete_candidates: 1
  KEEP: tmp_v4738122593335909120.mp4 2019-04-14T09:54:23.832120+08:00 /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/9/96ED2D63-5446-43F7-BED3-4B8E568BCC1A.mp4
  DELETE: tmp_v4738122593335909120.mp4 2019-04-14T09:55:36.001581+08:00 /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/D/D0DF33BE-8226-4B49-8937-3512A85A2D97.mp4
--------------------------------------------------------------------------------
reason: SAME_UNIQUE_ID_AND_SAME_SHA256
asset_count: 2
keep_assets: 1
delete_candidates: 1
  KEEP: IMG_0361.mp4 2021

In [7]:
# ============================================================
# Cell 7. Write permanent operation report
# ============================================================

REPORT_DIR.mkdir(parents=True, exist_ok=True)

report_result = write_operation_report(
    report_dir=REPORT_DIR,
    duplicate_analysis=duplicate_analysis,
    inventory=inventory,
    assets_without_unique_id=assets_without_unique_id,
    duplicate_candidate_groups=duplicate_candidate_groups,
    run_timestamp=RUN_TIMESTAMP,
    library_id=LIBRARY_ID,
    library_path=LIBRARY_PATH,
    inventory_cache_path=INVENTORY_CACHE_PATH,
)

delete_candidate_rows = report_result["delete_candidate_rows"]
keep_asset_rows = report_result["keep_asset_rows"]
duplicate_review_asset_rows = report_result["duplicate_review_asset_rows"]
location_conflict_rows = report_result["location_conflict_rows"]
live_photo_candidate_rows = report_result["live_photo_candidate_rows"]
assets_without_unique_id_rows = report_result["assets_without_unique_id_rows"]
status_counts = report_result["status_counts"]
safety_counts = report_result["safety_counts"]

print("Wrote report to:", REPORT_DIR)
print("delete_candidate_rows:", len(delete_candidate_rows))
print("keep_asset_rows:", len(keep_asset_rows))
print("duplicate_review_asset_rows:", len(duplicate_review_asset_rows))
print("location_conflict_rows:", len(location_conflict_rows))
print("live_photo_candidate_rows:", len(live_photo_candidate_rows))
print("assets_without_unique_id_rows:", len(assets_without_unique_id_rows))

Photos Library Duplicate Cleanup Report

run_timestamp: 20260608-202149
library_id: backup_20250317
library_path: /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary
inventory_cache_path: /Users/huohsien/workspace/python/explore_photos_library/data/inventory_cache/backup_20250317.inventory.pkl.gz

asset_count: 71596
assets_without_unique_id: 0
duplicate_candidate_group_count: 13
duplicate_candidate_asset_count: 26

status_counts:
{
  "DELETABLE_DUPLICATE": 13
}

safety_counts:
{
  "live_photo_candidate_group_count": 0,
  "location_conflict_group_count": 0,
  "unreadable_original_group_count": 0,
  "sha_error_group_count": 0
}

delete_candidate_asset_count: 13
keep_asset_count: 13
duplicate_review_asset_count: 26
location_conflict_row_count: 0
live_photo_candidate_row_count: 0
assets_without_unique_id_row_count: 0

Duplicate cleanup v1 decision rule:
- date_added is NOT part of photo_library_asset_uniq

In [8]:
# ============================================================
# Cell 8. Optional: print manual deletion list
# ============================================================
#
# This notebook does NOT delete anything from Photos Library.
# It only produces a report and delete candidate list.
#
# For actual deletion, review delete_candidates.tsv first.

for row in delete_candidate_rows[:100]:
    print(
        row["original_filename"],
        row["date"],
        row["date_added"],
        row["path"],
    )

if len(delete_candidate_rows) > 100:
    print("... more delete candidates not printed")


tmp_v4738122593335909120.mp4 2019-04-14T09:54:04.082973+08:00 2019-04-14T09:55:36.001581+08:00 /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/D/D0DF33BE-8226-4B49-8937-3512A85A2D97.mp4
IMG_0361.mp4 2020-07-27T00:37:58+08:00 2021-06-11T19:44:17.297069+08:00 /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/4/492D4033-C6AB-4F24-9023-F02F1144AB55.mp4
IMG_4980.MOV 2022-06-30T10:43:45+08:00 2023-11-05T16:54:59.163527+08:00 /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/C/CE2F2CD9-E74B-4E6C-9E07-5886F2A444E6.mov
ScreenRecording_10-13-2020 19-04-56_1.MP4 2020-10-13T19:04:57+08:00 2020-10-14T20:02:28.177526+08:00 /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）-

In [9]:
# ============================================================
# Compare duplicate groups between two report folders
# Find groups that disappeared after adjustment_signature fix
# ============================================================

import json
from pathlib import Path

OLD_REPORT_DIR = Path(
    "IMPORTANT_Photos_Library_Critical_Operation_Reports/"
    "20260608-195729__Photos_Library_Duplicate_Cleanup__backup_20250317"
)

NEW_REPORT_DIR = Path(
    "IMPORTANT_Photos_Library_Critical_Operation_Reports/"
    "20260608-202149__Photos_Library_Duplicate_Cleanup__backup_20250317"
)


def load_duplicate_analysis(report_dir):
    with open(report_dir / "duplicate_analysis.json", "r", encoding="utf-8") as f:
        return json.load(f)


def group_key(record):
    # Use UUID set as stable group identity.
    uuids = []

    for asset in record.get("assets") or []:
        uuid = asset.get("uuid")
        if uuid:
            uuids.append(uuid)

    return tuple(sorted(uuids))


old_records = load_duplicate_analysis(OLD_REPORT_DIR)
new_records = load_duplicate_analysis(NEW_REPORT_DIR)

old_by_key = {
    group_key(record): record
    for record in old_records
}

new_by_key = {
    group_key(record): record
    for record in new_records
}

disappeared_keys = sorted(set(old_by_key) - set(new_by_key))
new_keys = sorted(set(new_by_key) - set(old_by_key))

print("old group count:", len(old_by_key))
print("new group count:", len(new_by_key))
print("disappeared group count:", len(disappeared_keys))
print("new group count not in old:", len(new_keys))

print()
print("=" * 100)
print("Groups disappeared from old report")
print("=" * 100)

for index, key in enumerate(disappeared_keys, start=1):
    record = old_by_key[key]

    print()
    print("-" * 100)
    print("disappeared group:", index)
    print("old status:", record.get("status"))
    print("old reason:", record.get("reason"))

    for asset in record.get("assets") or []:
        print()
        print("uuid:", asset.get("uuid"))
        print("original_filename:", asset.get("original_filename"))
        print("date:", asset.get("date"))
        print("date_added:", asset.get("date_added"))
        print("path:", asset.get("path"))
        print("path_edited:", asset.get("path_edited"))
        print("hasadjustments:", asset.get("hasadjustments"))
        print("width:", asset.get("width"))
        print("height:", asset.get("height"))
        print("original_width:", asset.get("original_width"))
        print("original_height:", asset.get("original_height"))
        print("edited_duration_seconds:", asset.get("edited_duration_seconds"))
        print("adjustment_signature:", asset.get("adjustment_signature"))

old group count: 17
new group count: 13
disappeared group count: 4
new group count not in old: 0

Groups disappeared from old report

----------------------------------------------------------------------------------------------------
disappeared group: 1
old status: DELETABLE_DUPLICATE
old reason: SAME_UNIQUE_ID_AND_SAME_SHA256

uuid: 538849E9-4597-433D-8179-8748A7B2ECC9
original_filename: IMG_0245.JPG
date: 2022-11-11T22:43:53.965508+08:00
date_added: 2022-11-11T22:43:53.966928+08:00
path: /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/5/538849E9-4597-433D-8179-8748A7B2ECC9.jpeg
path_edited: None
hasadjustments: False
width: None
height: None
original_width: None
original_height: None
edited_duration_seconds: None
adjustment_signature: None

uuid: 0125A7C6-A997-4874-B0A6-1947339AB629
original_filename: IMG_0245.JPG
date: 2022-11-11T22:43:53.965508+08:00
date_added: 2022-11-13T04:54:03

In [12]:
# ============================================================
# TEMP CELL - Clean manual review notes for 4 disappeared groups
# Delete this cell after copying the output.
# ============================================================

TARGET_GROUPS = {
    "IMG_0245.JPG": [
        "538849E9-4597-433D-8179-8748A7B2ECC9",
        "0125A7C6-A997-4874-B0A6-1947339AB629",
    ],
    "PDF document-1.jpeg": [
        "237D1221-9F5D-4267-93F0-13752ADE8ED4",
        "6A2D2F9D-6095-4D8C-AE8B-1C82C5FFE124",
    ],
    "IMG_0234.PNG": [
        "799456F9-91E4-4859-90F3-198B1297C916",
        "5E51F8F4-6775-4851-A5C4-B297BE1597B5",
    ],
    "IMG_0106.JPG": [
        "70AD6AB0-90AE-4EE9-92B7-3EDE652C1406",
        "7E6F71B1-FAE6-4566-9066-BC501BC8F76A",
    ],
}


def _get_field(obj, *names):
    for name in names:
        if isinstance(obj, dict):
            value = obj.get(name)
        else:
            value = getattr(obj, name, None)

        if value is not None:
            return value

    return None


def _album_titles(asset):
    albums = asset.get("albums") or {}

    titles = []
    for album in albums.values():
        title = _get_field(album, "title", "name")
        if title:
            titles.append(str(title))

    return sorted(set(titles))


def _folder_paths(asset):
    folders = asset.get("folders") or {}

    paths = []
    for folder in folders.values():
        path = _get_field(folder, "path", "folder_path", "full_path")
        title = _get_field(folder, "title", "name")

        if path:
            paths.append(str(path))
        elif title:
            paths.append(str(title))

    return sorted(set(paths))


uuid_to_asset = {
    asset.get("uuid"): asset
    for asset in inventory["assets"]
}


print("Disappeared adjusted-image groups for manual Photos review")
print("Reason: These disappeared after adjustment_signature was fixed.")
print("Meaning: They were probably false duplicate candidates.")
print("Action: Check visually in Photos by image name + Date Added + Albums.")
print()

for group_index, (image_name, uuids) in enumerate(TARGET_GROUPS.items(), start=1):
    print("=" * 80)
    print(f"{group_index}. {image_name}")
    print("=" * 80)

    for asset_index, uuid in enumerate(uuids, start=1):
        asset = uuid_to_asset.get(uuid)

        print()
        print(f"Asset {asset_index}")

        if asset is None:
            print("  ERROR: asset not found in current inventory")
            continue

        print("  Date Added:", asset.get("date_added"))
        print("  Has Adjustments:", asset.get("hasadjustments"))

        if asset.get("hasadjustments"):
            print("  Adjustment Type:", asset.get("adjustment_type"))
            print("  External Edit:", asset.get("external_edit"))
            print("  UTI:", asset.get("uti"))
            print("  UTI Original:", asset.get("uti_original"))
            print("  UTI Edited:", asset.get("uti_edited"))
            print("  Adjustment Signature:", asset.get("adjustment_signature"))

        albums = _album_titles(asset)
        if albums:
            print("  Albums:")
            for title in albums:
                print("    -", title)

        folders = _folder_paths(asset)
        if folders:
            print("  Folder Paths:")
            for path in folders:
                print("    -", path)

    print()

Disappeared adjusted-image groups for manual Photos review
Reason: These disappeared after adjustment_signature was fixed.
Meaning: They were probably false duplicate candidates.
Action: Check visually in Photos by image name + Date Added + Albums.

1. IMG_0245.JPG

Asset 1
  Date Added: 2022-11-11T22:43:53.966928+08:00
  Has Adjustments: False
  Albums:
    - 2022年11月11日 周思慧 番外篇 今天突然想念她，上微信沒看到她的信息。覺得自己該放下了。但是忍不住又問她最近過得怎麼樣。然後我正在寫這段話的時候她好像回應了！ 唉！結果從10:46聊到11:08，22分鐘！我見好就收！

Asset 2
  Date Added: 2022-11-13T04:54:03.277919+08:00
  Has Adjustments: True
  Adjustment Type: 2
  External Edit: False
  UTI: public.jpeg
  UTI Original: public.jpeg
  UTI Edited: public.jpeg
  Adjustment Signature: (670, 1081)
  Albums:
    - 2022年11月11日 周思慧 番外篇 今天突然想念她，上微信沒看到她的信息。覺得自己該放下了。但是忍不住又問她最近過得怎麼樣。然後我正在寫這段話的時候她好像回應了！ 唉！結果從10:46聊到11:08，22分鐘！我見好就收！

2. PDF document-1.jpeg

Asset 1
  Date Added: 2021-04-17T17:15:03.404507+08:00
  Has Adjustments: True
  Adjustment Type: 2
  External Edit: False
  UTI: publi